In [1]:
import pandas as pd
import mysql.connector
import socket
from typing import Optional, List
import pymysql
from DATA.stock_invest_function import fetch_table_data, get_db_host

In [2]:

def get_revenue_pivot_by_report_date(
    db_info: dict,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    tickers: Optional[List[str]] = None,
    quarter_type: Optional[str] = None
) -> pd.DataFrame:
    """
    DART 데이터에서 매출액 데이터를 추출하여 pivot 테이블 생성
    index: report_date, columns: ticker, values: thstrm_amount

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    start_date : str, optional
        시작 날짜 (형식: 'YYYY-MM-DD')
    end_date : str, optional
        종료 날짜 (형식: 'YYYY-MM-DD')
    tickers : List[str], optional
        특정 ticker 리스트 (6자리 코드)
    quarter_type : str, optional
        분기 타입 ('Q1', 'Q3', 'H1', 'FY')

    Returns:
    --------
    pd.DataFrame
        index=report_date, columns=ticker인 pivot 테이블
    """

    conn = mysql.connector.connect(**db_info)

    try:
        # SQL 쿼리
        query = """
        SELECT
            report_date,
            ticker,
            quarter,
            account_id,
            thstrm_amount
        FROM korea_fs_data_from_DART
        WHERE account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
          AND thstrm_amount IS NOT NULL
          AND report_date IS NOT NULL
        """

        params = []

        # 날짜 필터
        if start_date:
            query += " AND report_date >= %s"
            params.append(start_date)
        if end_date:
            query += " AND report_date <= %s"
            params.append(end_date)

        # ticker 필터
        if tickers:
            placeholders = ','.join(['%s'] * len(tickers))
            query += f" AND ticker IN ({placeholders})"
            params.extend(tickers)

        # 분기 필터
        if quarter_type:
            query += " AND quarter = %s"
            params.append(quarter_type)

        query += " ORDER BY report_date, ticker"

        # 쿼리 실행
        df = pd.read_sql(query, conn, params=params if params else None)

        if len(df) == 0:
            print("조건에 맞는 데이터가 없습니다.")
            return pd.DataFrame()

        print(f"\n{'='*60}")
        print(f"원본 데이터: {len(df):,}행")
        print(f"기간: {df['report_date'].min()} ~ {df['report_date'].max()}")
        print(f"종목(ticker) 수: {df['ticker'].nunique()}개")
        print(f"리포트 날짜 수: {df['report_date'].nunique()}개")
        print(f"{'='*60}\n")

        # 같은 report_date-ticker에 여러 account_id가 있는 경우 평균 사용
        # (ifrs_Revenue와 ifrs-full_Revenue가 둘 다 있는 경우)
        df_agg = df.groupby(['report_date', 'ticker'])['thstrm_amount'].mean().reset_index()

        # pivot 테이블 생성
        pivot_df = df_agg.pivot(index='report_date', columns='ticker', values='thstrm_amount')

        # 인덱스를 날짜 형식으로 변환
        pivot_df.index = pd.to_datetime(pivot_df.index)

        # 날짜순 정렬
        pivot_df = pivot_df.sort_index()

        print(f"Pivot 테이블 생성 완료:")
        print(f"  - 행(report_date): {len(pivot_df)}개")
        print(f"  - 열(ticker): {len(pivot_df.columns)}개")
        print(f"  - 전체 셀 수: {len(pivot_df) * len(pivot_df.columns):,}개")
        print(f"  - 결측값: {pivot_df.isna().sum().sum():,}개")
        print(f"  - 결측값 비율: {pivot_df.isna().sum().sum() / (len(pivot_df) * len(pivot_df.columns)) * 100:.2f}%\n")

        return pivot_df

    finally:
        conn.close()


def get_revenue_pivot_summary(
    db_info: dict,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    tickers: Optional[List[str]] = None,
    quarter_type: Optional[str] = None,
    fill_method: Optional[str] = None
) -> pd.DataFrame:
    """
    매출액 pivot 테이블 생성 + 기초 통계 및 결측값 처리

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    start_date : str, optional
        시작 날짜
    end_date : str, optional
        종료 날짜
    tickers : List[str], optional
        특정 ticker 리스트
    quarter_type : str, optional
        분기 타입
    fill_method : str, optional
        결측값 처리 방법 ('ffill', 'bfill', None)

    Returns:
    --------
    pd.DataFrame
        pivot 테이블 (결측값 처리 적용)
    """

    # 기본 pivot 생성
    pivot_df = get_revenue_pivot_by_report_date(
        db_info=db_info,
        start_date=start_date,
        end_date=end_date,
        tickers=tickers,
        quarter_type=quarter_type
    )

    if len(pivot_df) == 0:
        return pivot_df

    # 결측값 처리
    if fill_method == 'ffill':
        pivot_df = pivot_df.fillna(method='ffill')
        print("결측값을 forward fill로 처리했습니다.")
    elif fill_method == 'bfill':
        pivot_df = pivot_df.fillna(method='bfill')
        print("결측값을 backward fill로 처리했습니다.")

    # 기초 통계
    print("\n=== 기초 통계 (ticker별 평균 매출액) ===")
    ticker_avg = pivot_df.mean().sort_values(ascending=False)
    print(ticker_avg.head(10))

    return pivot_df


def analyze_revenue_growth(
    pivot_df: pd.DataFrame,
    ticker: str,
    periods: int = 4
) -> pd.DataFrame:
    """
    특정 ticker의 매출액 성장률 분석

    Parameters:
    -----------
    pivot_df : pd.DataFrame
        매출액 pivot 테이블
    ticker : str
        분석할 ticker
    periods : int
        성장률 계산 기간 (분기 수)

    Returns:
    --------
    pd.DataFrame
        매출액 및 성장률 데이터프레임
    """

    if ticker not in pivot_df.columns:
        print(f"Ticker {ticker}가 데이터에 없습니다.")
        return pd.DataFrame()

    # 해당 ticker 데이터 추출
    revenue = pivot_df[ticker].dropna()

    # 성장률 계산
    result = pd.DataFrame({
        'report_date': revenue.index,
        'revenue': revenue.values,
        f'growth_{periods}q': revenue.pct_change(periods=periods) * 100
    })

    result.set_index('report_date', inplace=True)

    print(f"\n=== Ticker {ticker} 매출액 분석 ===")
    print(f"기간: {result.index.min().strftime('%Y-%m-%d')} ~ {result.index.max().strftime('%Y-%m-%d')}")
    print(f"평균 매출액: {result['revenue'].mean():,.0f}")
    print(f"최근 매출액: {result['revenue'].iloc[-1]:,.0f}")
    print(f"평균 {periods}분기 성장률: {result[f'growth_{periods}q'].mean():.2f}%")
    print(f"\n최근 5개 데이터:")
    print(result.tail())

    return result

def get_indicator_pivot(
    db_info: dict,
    indicator: str,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    tickers: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    특정 indicator의 값을 pivot 테이블로 생성
    index: date, columns: ticker, values: value

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    indicator : str
        지표명 (예: 'ETS', 'Ensemble', 'SARIMA', 'Theta')
    start_date : str, optional
        시작 날짜 (형식: 'YYYY-MM-DD')
    end_date : str, optional
        종료 날짜 (형식: 'YYYY-MM-DD')
    tickers : List[str], optional
        특정 ticker 리스트

    Returns:
    --------
    pd.DataFrame
        index=date, columns=ticker인 pivot 테이블
    """

    conn = mysql.connector.connect(**db_info)

    try:
        # 테이블명을 확인해야 합니다 (예시에서는 forecast_results로 가정)
        query = """
        SELECT
            date,
            ticker,
            indicator,
            value
        FROM korea_revenue_forecast_result
        WHERE indicator = %s
          AND value IS NOT NULL
        """

        params = [indicator]

        # 날짜 필터
        if start_date:
            query += " AND date >= %s"
            params.append(start_date)
        if end_date:
            query += " AND date <= %s"
            params.append(end_date)

        # ticker 필터
        if tickers:
            placeholders = ','.join(['%s'] * len(tickers))
            query += f" AND ticker IN ({placeholders})"
            params.extend(tickers)

        query += " ORDER BY date, ticker"

        # 쿼리 실행
        df = pd.read_sql(query, conn, params=params)

        if len(df) == 0:
            print(f"조건에 맞는 데이터가 없습니다. (indicator: {indicator})")
            return pd.DataFrame()

        print(f"\n{'='*60}")
        print(f"지표(Indicator): {indicator}")
        print(f"원본 데이터: {len(df):,}행")
        print(f"기간: {df['date'].min()} ~ {df['date'].max()}")
        print(f"종목(ticker) 수: {df['ticker'].nunique()}개")
        print(f"날짜 수: {df['date'].nunique()}개")
        print(f"{'='*60}\n")

        # 같은 date-ticker에 여러 값이 있는 경우 평균 사용 (일반적으로 없어야 함)
        df_agg = df.groupby(['date', 'ticker'])['value'].mean().reset_index()

        # pivot 테이블 생성
        pivot_df = df_agg.pivot(index='date', columns='ticker', values='value')

        # 인덱스를 날짜 형식으로 변환
        pivot_df.index = pd.to_datetime(pivot_df.index)

        # 날짜순 정렬
        pivot_df = pivot_df.sort_index()

        print(f"Pivot 테이블 생성 완료:")
        print(f"  - 행(date): {len(pivot_df)}개")
        print(f"  - 열(ticker): {len(pivot_df.columns)}개")
        print(f"  - 전체 셀 수: {len(pivot_df) * len(pivot_df.columns):,}개")
        print(f"  - 결측값: {pivot_df.isna().sum().sum():,}개")
        print(f"  - 결측값 비율: {pivot_df.isna().sum().sum() / (len(pivot_df) * len(pivot_df.columns)) * 100:.2f}%\n")

        return pivot_df

    finally:
        conn.close()


def get_multiple_indicators_pivot(
    db_info: dict,
    indicators: List[str],
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    tickers: Optional[List[str]] = None
) -> dict:
    """
    여러 indicator의 pivot 테이블을 한 번에 생성

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    indicators : List[str]
        지표명 리스트 (예: ['ETS', 'Ensemble', 'SARIMA', 'Theta'])
    start_date : str, optional
        시작 날짜
    end_date : str, optional
        종료 날짜
    tickers : List[str], optional
        특정 ticker 리스트

    Returns:
    --------
    dict
        {indicator: pivot_df} 형태의 딕셔너리
    """

    result = {}

    for indicator in indicators:
        print(f"\n{'#'*60}")
        print(f"# {indicator} 처리 중...")
        print(f"{'#'*60}")

        pivot_df = get_indicator_pivot(
            db_info=db_info,
            indicator=indicator,
            start_date=start_date,
            end_date=end_date,
            tickers=tickers
        )

        result[indicator] = pivot_df

    return result


def compare_indicators_for_ticker(
    db_info: dict,
    ticker: str,
    indicators: List[str],
    start_date: Optional[str] = None,
    end_date: Optional[str] = None
) -> pd.DataFrame:
    """
    특정 ticker에 대해 여러 indicator 값을 비교

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    ticker : str
        종목 코드
    indicators : List[str]
        지표명 리스트
    start_date : str, optional
        시작 날짜
    end_date : str, optional
        종료 날짜

    Returns:
    --------
    pd.DataFrame
        index=date, columns=indicator인 비교 테이블
    """

    conn = mysql.connector.connect(**db_info)

    try:
        query = """
        SELECT
            date,
            ticker,
            indicator,
            value
        FROM forecast_results
        WHERE ticker = %s
          AND value IS NOT NULL
        """

        params = [ticker]

        if start_date:
            query += " AND date >= %s"
            params.append(start_date)
        if end_date:
            query += " AND date <= %s"
            params.append(end_date)

        if indicators:
            placeholders = ','.join(['%s'] * len(indicators))
            query += f" AND indicator IN ({placeholders})"
            params.extend(indicators)

        query += " ORDER BY date, indicator"

        df = pd.read_sql(query, conn, params=params)

        if len(df) == 0:
            print(f"Ticker {ticker}에 대한 데이터가 없습니다.")
            return pd.DataFrame()

        # pivot: date x indicator
        pivot_df = df.pivot(index='date', columns='indicator', values='value')
        pivot_df.index = pd.to_datetime(pivot_df.index)
        pivot_df = pivot_df.sort_index()

        print(f"\n{'='*60}")
        print(f"Ticker {ticker} - Indicator 비교")
        print(f"기간: {pivot_df.index.min().strftime('%Y-%m-%d')} ~ {pivot_df.index.max().strftime('%Y-%m-%d')}")
        print(f"Indicators: {list(pivot_df.columns)}")
        print(f"{'='*60}\n")

        return pivot_df

    finally:
        conn.close()


def get_indicator_statistics(
    pivot_df: pd.DataFrame,
    ticker: str
) -> pd.Series:
    """
    특정 ticker의 통계 정보 계산

    Parameters:
    -----------
    pivot_df : pd.DataFrame
        pivot 테이블
    ticker : str
        종목 코드

    Returns:
    --------
    pd.Series
        통계 정보
    """

    if ticker not in pivot_df.columns:
        print(f"Ticker {ticker}가 데이터에 없습니다.")
        return pd.Series()

    stats = pivot_df[ticker].describe()

    print(f"\n=== Ticker {ticker} 통계 ===")
    print(stats)
    print(f"\n최근 5개 값:")
    print(pivot_df[ticker].tail())

    return stats


def adjust_fy_to_q4(df: pd.DataFrame) -> pd.DataFrame:
    """
    FY(연간 누적) 데이터를 순수 Q4로 변환
    Q4 = FY - (Q1 + Q2 + Q3)
    """
    result_df = df.copy()

    for year in result_df['bsns_year'].unique():
        year_mask = result_df['bsns_year'] == year

        # FY 행 (현재 4분기 누적)
        fy_mask = year_mask & (result_df['quarter'] == 'FY')

        if fy_mask.any():
            fy_amount = result_df.loc[fy_mask, 'thstrm_amount'].iloc[0]

            # Q1+Q2+Q3 합
            q123_mask = year_mask & result_df['quarter'].isin(['Q1', 'Q2', 'Q3'])
            q123_sum = result_df.loc[q123_mask, 'thstrm_amount'].sum()

            pure_q4 = fy_amount - q123_sum

            # FY → Q4 로 바꾸고 금액도 순수 Q4 로 수정
            result_df.loc[fy_mask, 'quarter'] = 'Q4'
            result_df.loc[fy_mask, 'thstrm_amount'] = pure_q4

    return result_df

def pivot_to_long_with_meta(pivot_df: pd.DataFrame) -> pd.DataFrame:
    df = pivot_df.copy()
    df.index = pd.to_datetime(df.index)

    # wide → long : date, ticker, thstrm_amount
    long_df = df.stack().reset_index()
    long_df.columns = ['date', 'ticker', 'thstrm_amount']

    # 연도 / 분기 정보 생성
    long_df['bsns_year'] = long_df['date'].dt.year

    # 월 → 분기 라벨(Q1, Q2, Q3, FY) 매핑
    month = long_df['date'].dt.month
    quarter_map = {3: 'Q1', 6: 'Q2', 9: 'Q3', 12: 'FY'}
    long_df['quarter'] = month.map(quarter_map)

    return long_df

def adjust_pivot_q4_from_cumulative(pivot_df: pd.DataFrame) -> pd.DataFrame:
    # 1) wide → long + 메타컬럼
    long_df = pivot_to_long_with_meta(pivot_df)

    # 2) ticker 별로 FY→Q4 보정
    adjusted_long = (
        long_df
        .groupby('ticker', group_keys=False)
        .apply(adjust_fy_to_q4)
    )

    # 3) 다시 wide 형태로 피벗
    adjusted_pivot = (
        adjusted_long
        .pivot(index='date', columns='ticker', values='thstrm_amount')
        .sort_index()
    )

    return adjusted_pivot

def get_indicator_pivot_by_created_at(
    db_info: dict,
    indicator: str,
    created_date: str,   # YYYY-MM-DD
    table_name: str = "korea_revenue_forecast_result",
    start_date: Optional[str] = None,
    end_date: Optional[str] = None
) -> pd.DataFrame:
    """
    created_at 기준(run 기준일)으로 indicator 예측 데이터를 pivot_table로 반환
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],
        charset="utf8mb4"
    )

    try:
        print("=" * 70)
        print(f"📊 Indicator Pivot 조회 (created_at 기준)")
        print("=" * 70)
        print(f"✓ table       : {table_name}")
        print(f"✓ indicator   : {indicator}")
        print(f"✓ created_date: {created_date}")
        if start_date:
            print(f"✓ start date  : {start_date}")
        if end_date:
            print(f"✓ end date    : {end_date}")
        print("=" * 70)

        params = [created_date, indicator]

        date_filter = ""
        if start_date:
            date_filter += " AND date >= %s"
            params.append(start_date)
        if end_date:
            date_filter += " AND date <= %s"
            params.append(end_date)

        sql = f"""
        SELECT
            date,
            ticker,
            value
        FROM {table_name}
        WHERE DATE(created_at) = %s
          AND indicator = %s
          {date_filter}
        ORDER BY date, ticker
        """

        df = pd.read_sql(sql, conn, params=params)

        if df.empty:
            print("❌ 조회 결과 없음")
            return pd.DataFrame()

        print(f"✅ {len(df):,} rows loaded")

        # 날짜 정리
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

        pivot_df = df.pivot_table(
            index="date",
            columns="ticker",
            values="value",
            aggfunc="first"
        )

        pivot_df.columns.name = None

        print("=" * 70)
        print("📊 Pivot 생성 완료")
        print(f"✓ dates  : {len(pivot_df):,}")
        print(f"✓ tickers: {len(pivot_df.columns):,}")
        print(f"✓ range  : {pivot_df.index.min()} ~ {pivot_df.index.max()}")
        print("=" * 70)

        return pivot_df

    finally:
        conn.close()


def get_all_indicators_pivot(
    db_info: dict,
    input_date: str,
    table_name: str = "korea_revenue_forecast_result",
    indicators: Optional[List[str]] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None
) -> pd.DataFrame:
    """
    모든 indicator를 wide format으로 결합하여 반환

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보
    input_date : str
        조회할 예측 입력 날짜
    table_name : str
        테이블 이름
    indicators : List[str], optional
        조회할 지표 리스트 (기본값: ['SARIMA', 'ETS', 'Theta', 'Ensemble'])
    start_date : str, optional
        시작 날짜 필터 (forecast_date 기준)
    end_date : str, optional
        종료 날짜 필터 (forecast_date 기준)

    Returns:
    --------
    pd.DataFrame
        모든 지표가 결합된 wide format DataFrame
        - index: forecast_date, ticker
        - columns: SARIMA, ETS, Theta, Ensemble
    """

    # 기본 지표 설정
    if indicators is None:
        indicators = ['SARIMA', 'ETS', 'Theta', 'Ensemble']

    # DB 연결
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],
        charset="utf8mb4"
    )

    try:
        print("=" * 70)
        print("📊 전체 지표 데이터 조회")
        print("=" * 70)
        print(f"✓ input_date: {input_date}")
        print(f"✓ 지표: {', '.join(indicators)}")
        print("=" * 70)

        # 각 지표별로 데이터 조회
        all_dfs = []

        for indicator in indicators:
            date_filter = ""
            params = [input_date, indicator]

            if start_date:
                date_filter += " AND forecast_date >= %s"
                params.append(start_date)

            if end_date:
                date_filter += " AND forecast_date <= %s"
                params.append(end_date)

            sql = f"""
            SELECT
                forecast_date as date,
                ticker,
                indicator,
                value
            FROM {table_name}
            WHERE DATE(input_date) = DATE(%s)
                AND indicator = %s
                {date_filter}
            ORDER BY ticker, forecast_date
            """

            df = pd.read_sql(sql, conn, params=params)

            if not df.empty:
                print(f"  ✓ {indicator}: {len(df):,}행 조회")
                all_dfs.append(df)
            else:
                print(f"  ⚠️ {indicator}: 데이터 없음")

        if not all_dfs:
            print("\n❌ 조회된 데이터가 없습니다.")
            return pd.DataFrame()

        # 모든 데이터 결합
        combined_df = pd.concat(all_dfs, axis=0, ignore_index=True)
        print(f"\n✅ 총 {len(combined_df):,}행 결합 완료")

        # 날짜 형식 정리
        combined_df['date'] = pd.to_datetime(combined_df['date']).dt.strftime('%Y-%m-%d')

        # pivot_table 생성
        pivot_df = combined_df.pivot_table(
            index=['date', 'ticker'],
            columns='indicator',
            values='value',
            aggfunc='first'
        )

        # 컬럼 이름 정리
        pivot_df.columns.name = None

        # index를 컬럼으로 변환
        pivot_df = pivot_df.reset_index()

        # 컬럼 순서 정리
        column_order = ['date', 'ticker'] + [col for col in indicators if col in pivot_df.columns]
        pivot_df = pivot_df[column_order]

        print("\n" + "=" * 70)
        print("📊 전체 지표 pivot_table 생성 완료")
        print("=" * 70)
        print(f"✓ 행 개수: {len(pivot_df):,}")
        print(f"✓ ticker 개수: {pivot_df['ticker'].nunique():,}")
        print(f"✓ 날짜 범위: {pivot_df['date'].min()} ~ {pivot_df['date'].max()}")
        print(f"✓ 컬럼: {', '.join(pivot_df.columns.tolist())}")
        print("=" * 70)

        return pivot_df

    finally:
        conn.close()

def get_unique_created_dates(
    db_info: dict,
    table_name: str = "korea_revenue_forecast_result",
    limit: Optional[int] = None,   # ✅ Python 3.9 호환
    desc: bool = True
) -> pd.DataFrame:
    """
    created_at의 날짜(YYYY-mm-dd)만 추출해서 unique 날짜 목록 반환
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=int(db_info["port"]),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],
        charset="utf8mb4"
    )
    try:
        order = "DESC" if desc else "ASC"
        lim = f"LIMIT {int(limit)}" if limit is not None else ""

        sql = f"""
        SELECT DATE(created_at) AS created_date
        FROM {table_name}
        GROUP BY DATE(created_at)
        ORDER BY created_date {order}
        {lim}
        """
        df = pd.read_sql(sql, conn)

        print("=" * 60)
        print("📅 created_at 기준 unique 날짜(YYYY-mm-dd)")
        print("=" * 60)
        print(df.to_string(index=False))
        print("=" * 60)

        return df
    finally:
        conn.close()

In [3]:
from DATA.stock_invest_function import fetch_table_data, get_db_host

# 데이터베이스 연결 정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}


# 예시 1: 전체 매출액 pivot 생성
print("\n" + "="*80)
print("예시 1: 전체 매출액 pivot 테이블")
print("="*80)
pivot_all = get_revenue_pivot_by_report_date(
    db_info=db_info,
    start_date='2020-01-01',
    end_date='2025-09-30'
)
print(pivot_all.head(10))
print(f"\n데이터 형태: {pivot_all.shape}")


예시 1: 전체 매출액 pivot 테이블

원본 데이터: 44,216행
기간: 2020-03-31 ~ 2025-09-30
종목(ticker) 수: 2366개
리포트 날짜 수: 23개

Pivot 테이블 생성 완료:
  - 행(report_date): 23개
  - 열(ticker): 2366개
  - 전체 셀 수: 54,418개
  - 결측값: 10,202개
  - 결측값 비율: 18.75%

ticker             000020        000040        000050        000070  \
report_date                                                           
2020-03-31   6.700367e+10  2.855757e+10  7.399416e+10  5.973060e+11   
2020-06-30   6.983330e+10  3.120514e+10  8.620966e+10  5.794880e+11   
2020-09-30   6.656821e+10  2.908280e+10  8.432171e+10  6.450780e+11   
2020-12-31   2.720754e+11  1.178344e+11  3.292247e+11  2.471226e+12   
2021-03-31   7.182724e+10  2.270011e+10  8.257075e+10  7.006650e+11   
2021-06-30   7.580235e+10  3.799214e+10  9.396038e+10  7.852480e+11   
2021-09-30   6.907114e+10  3.450467e+10  9.347672e+10  8.214320e+11   
2021-12-31   2.930181e+11  1.335104e+11  3.833571e+11  3.107313e+12   
2022-03-31   8.529413e+10  2.915054e+10  9.678620e+10  8.222970e+11

In [4]:
unique_created_dates_df = get_unique_created_dates(db_info, limit=50)

📅 created_at 기준 unique 날짜(YYYY-mm-dd)
created_date
  2025-12-05
  2025-12-04


In [5]:
pivot_all_corrected = adjust_pivot_q4_from_cumulative(pivot_all)

# 확인
pivot_all_corrected.tail()

ticker,000020,000040,000050,000070,000080,000100,000120,000140,000150,000180,...,900250,900260,900270,900290,950130,950160,950170,950190,950210,950220
date,,,,,,,,,,,,,,,,,,,,,
2024-09-30,1.101934e+11,1.802475e+10,9.384048e+10,8.891899e+11,6.857402e+11,5.987849e+11,2.975814e+12,6.820442e+11,3.878513e+12,3.399984e+10,...,139519518.0,643055396.0,NaN,1.056165e+09,16038501.0,1551005.0,8.165059e+09,14762193.0,NaN,806272.0
2024-12-31,1.206335e+11,-3.564820e+10,1.104725e+11,8.755712e+11,6.270541e+11,4.961007e+11,3.160340e+12,6.230329e+11,5.199803e+12,3.193154e+10,...,132987967.0,669438960.0,NaN,9.196749e+08,21394325.0,545235.0,-9.866677e+09,14361878.0,NaN,-681355.0
2025-03-31,1.256744e+11,2.943066e+09,1.022726e+11,8.659359e+11,6.127769e+11,4.915640e+11,2.992586e+12,6.096430e+11,4.298676e+12,3.245234e+10,...,112775664.0,618365140.0,NaN,NaN,14942592.0,886776.0,1.004846e+10,14254089.0,NaN,9111.0
2025-06-30,1.250246e+11,4.536488e+09,9.510261e+10,9.075142e+11,6.465543e+11,5.789870e+11,3.048408e+12,6.428958e+11,5.346439e+12,3.264197e+10,...,147807587.0,655321703.0,NaN,NaN,962831.0,940892.0,7.875851e+09,14537947.0,NaN,17579.0
2025-09-30,1.219990e+11,3.408911e+09,9.690799e+10,8.896320e+11,6.695325e+11,5.700136e+11,3.066608e+12,6.658553e+11,4.452365e+12,3.135917e+10,...,NaN,NaN,NaN,1.070464e+09,2517049.0,1214461.0,NaN,15089520.0,NaN,12976.0


In [6]:

# 예시 1: 단일 indicator pivot 생성
print("\n" + "="*80)
print("예시 1: ETS indicator pivot 테이블")
print("="*80)

pivot_ets_day1 = get_indicator_pivot_by_created_at(
    db_info=db_info,
    indicator="ETS",
    created_date="2025-12-04",
    start_date="2012-12-01",
    end_date="2027-12-31"
)

print(pivot_ets_day1.head(10))
print(f"\n데이터 형태: {pivot_ets_day1.shape}")

pivot_ets_day2 = get_indicator_pivot_by_created_at(
    db_info=db_info,
    indicator="ETS",
    created_date="2025-12-05",
    start_date="2012-12-01",
    end_date="2027-12-31"
)

print(pivot_ets_day2.head(10))
print(f"\n데이터 형태: {pivot_ets_day2.shape}")


# # 예시 2: 특정 ticker만 조회
# print("\n" + "="*80)
# print("예시 2: 특정 ticker의 SARIMA 예측")
# print("="*80)
# pivot_sarima = get_indicator_pivot(
#     db_info=db_info,
#     indicator='SARIMA',
#     start_date='2025-01-01',
#     tickers=['000020', '000040']
# )
# print(pivot_sarima)
#
# # 예시 3: 여러 indicator 동시 조회
# print("\n" + "="*80)
# print("예시 3: 여러 indicator 동시 조회")
# print("="*80)
# indicators = ['ETS', 'Ensemble', 'SARIMA', 'Theta']
# pivots = get_multiple_indicators_pivot(
#     db_info=db_info,
#     indicators=indicators,
#     start_date='2025-01-01',
#     end_date='2027-12-31',
#     tickers=['000020']
# )
#
# # 각 indicator별 결과 확인
# for indicator, pivot_df in pivots.items():
#     print(f"\n{indicator} 결과:")
#     print(pivot_df.head())
#
# # 예시 4: 특정 ticker에 대한 indicator 비교
# print("\n" + "="*80)
# print("예시 4: Ticker 000020의 모든 indicator 비교")
# print("="*80)
# comparison = compare_indicators_for_ticker(
#     db_info=db_info,
#     ticker='000020',
#     indicators=['ETS', 'Ensemble', 'SARIMA', 'Theta'],
#     start_date='2025-01-01'
# )
# print(comparison.head(10))
#
# # 예시 5: 통계 분석
# print("\n" + "="*80)
# print("예시 5: 통계 분석")
# print("="*80)
# if '000020' in pivot_ets.columns:
#     stats = get_indicator_statistics(pivot_ets, '000020')
#
# # 예시 6: CSV 저장
# print("\n" + "="*80)
# print("예시 6: CSV 저장")
# print("="*80)
# pivot_ets.to_csv('forecast_ets_pivot.csv', encoding='utf-8-sig')
# print("저장 완료: forecast_ets_pivot.csv")
#
# # 여러 indicator를 각각 저장
# for indicator, pivot_df in pivots.items():
#     filename = f'forecast_{indicator.lower()}_pivot.csv'
#     pivot_df.to_csv(filename, encoding='utf-8-sig')
#     print(f"저장 완료: {filename}")
#
# # 예시 7: 특정 날짜의 예측값 비교
# print("\n" + "="*80)
# print("예시 7: 특정 날짜의 예측값 Top 10")
# print("="*80)
# if len(pivot_ets) > 0:
#     target_date = pivot_ets.index[10]  # 11번째 날짜
#     forecast_values = pivot_ets.loc[target_date].dropna().sort_values(ascending=False)
#     print(f"\n{target_date.strftime('%Y-%m-%d')} 예측값 Top 10:")
#     print(forecast_values.head(10))
#
# # 예시 8: 특정 ticker의 시계열 추출
# print("\n" + "="*80)
# print("예시 8: 특정 ticker 시계열 데이터")
# print("="*80)
# if '000020' in pivot_ets.columns:
#     timeseries = pivot_ets['000020'].dropna()
#     print(f"\nTicker 000020 - ETS 예측:")
#     print(timeseries)
#     print(f"\n평균: {timeseries.mean():,.2f}")
#     print(f"표준편차: {timeseries.std():,.2f}")


예시 1: ETS indicator pivot 테이블
📊 Indicator Pivot 조회 (created_at 기준)
✓ table       : korea_revenue_forecast_result
✓ indicator   : ETS
✓ created_date: 2025-12-04
✓ start date  : 2012-12-01
✓ end date    : 2027-12-31
✅ 3,598 rows loaded
📊 Pivot 생성 완료
✓ dates  : 29
✓ tickers: 400
✓ range  : 2025-06-30 ~ 2027-12-31
            000020  000040  000050  000070  000080  000100  000120  000140  \
date                                                                         
2025-06-30     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-01     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-02     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-03     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-04     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-05     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2025-07-06     NaN     NaN     NaN     NaN     NaN     NaN     

In [7]:
# 1) index를 DatetimeIndex로 통일 (지금 pivot은 문자열 index일 가능성이 높음)
pivot_ets_day1.index = pd.to_datetime(pivot_ets_day1.index)
pivot_ets_day2.index = pd.to_datetime(pivot_ets_day2.index)

# 2) day1, day2 가로 결합 (컬럼 충돌 방지: MultiIndex 컬럼)
combined = pd.concat(
    [pivot_ets_day1, pivot_ets_day2],
    axis=1,
    )

# 3) 분기말(3/6/9/12월 말)만 남기기
# - 월이 3,6,9,12 이고
# - 해당 월의 마지막 날인지 체크
idx = combined.index
is_q_end = idx.month.isin([3, 6, 9, 12]) & (idx.is_month_end)

combined_qend = combined.loc[is_q_end].sort_index()

print(combined_qend.head())
print("shape:", combined_qend.shape)

                  000020        000040        000050        000070  \
date                                                                 
2025-06-30           NaN           NaN           NaN           NaN   
2025-09-30           NaN           NaN           NaN           NaN   
2025-12-31  1.267333e+11 -6.172097e+09  1.035138e+11  8.972449e+11   
2026-03-31  1.376158e+11 -2.338416e+09  1.027085e+11  8.969599e+11   
2026-06-30  1.349638e+11  4.829497e+09  1.036341e+11  9.384721e+11   

                  000080        000100        000120        000140  \
date                                                                 
2025-06-30           NaN           NaN           NaN           NaN   
2025-09-30           NaN           NaN           NaN           NaN   
2025-12-31  6.283006e+11  5.136847e+11  3.187567e+12  5.963294e+11   
2026-03-31  6.240813e+11  5.111522e+11  3.003122e+12  6.053125e+11   
2026-06-30  6.680949e+11  5.932765e+11  3.130086e+12  6.789107e+11   

                  

In [8]:
combined_qend[['000660', '005930']]

,000660,005930
date,,
2025-06-30,NaN,NaN
2025-09-30,NaN,NaN
2025-12-31,2.684272e+13,8.355652e+13
2026-03-31,2.850416e+13,8.315349e+13
2026-06-30,3.400135e+13,8.204488e+13
2026-09-30,3.575013e+13,8.849932e+13
2026-12-31,3.839652e+13,8.675352e+13
2027-03-31,4.005795e+13,8.635049e+13
2027-06-30,4.555515e+13,8.524188e+13


In [9]:
forecast_revenue = combined_qend.loc['2025-12':]
forecast_revenue

,000020,000040,000050,000070,000080,000100,000120,000140,000150,000180,...,226320,234080,237690,237750,244920,252500,260660,277410,355150,950130
date,,,,,,,,,,,,,,,,,,,,,
2025-12-31,1.267333e+11,-6.172097e+09,1.035138e+11,8.972449e+11,6.283006e+11,5.136847e+11,3.187567e+12,5.963294e+11,4.703187e+12,3.200386e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-31,1.376158e+11,-2.338416e+09,1.027085e+11,8.969599e+11,6.240813e+11,5.111522e+11,3.003122e+12,6.053125e+11,4.603832e+12,3.020429e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-30,1.349638e+11,4.829497e+09,1.036341e+11,9.384721e+11,6.680949e+11,5.932765e+11,3.130086e+12,6.789107e+11,4.723224e+12,3.018428e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09-30,1.314093e+11,2.088317e+09,9.904246e+10,9.311861e+11,6.856323e+11,6.188431e+11,3.154965e+12,6.641743e+11,4.592868e+12,2.926547e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-12-31,1.365485e+11,-7.652214e+09,1.054152e+11,9.322036e+11,6.456663e+11,5.362445e+11,3.270253e+12,6.000458e+11,4.698936e+12,2.987358e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2027-03-31,1.474311e+11,-3.818533e+09,1.046098e+11,9.319186e+11,6.414471e+11,5.337120e+11,3.085808e+12,6.090289e+11,4.599581e+12,2.807401e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2027-06-30,1.447790e+11,3.349380e+09,1.055355e+11,9.734308e+11,6.854606e+11,6.158363e+11,3.212772e+12,6.826272e+11,4.718972e+12,2.805400e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2027-09-30,1.412246e+11,6.082001e+08,1.009439e+11,9.661448e+11,7.029980e+11,6.414029e+11,3.237651e+12,6.678907e+11,4.588617e+12,2.713520e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2027-12-31,1.463638e+11,-9.132331e+09,1.073166e+11,9.671623e+11,6.630321e+11,5.588044e+11,3.352939e+12,6.037623e+11,4.694685e+12,2.774330e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
revenue_actual_forecast = pd.concat([pivot_all_corrected, forecast_revenue])
revenue_actual_forecast

,000020,000040,000050,000070,000080,000100,000120,000140,000150,000180,...,081150,087600,099410,109820,115610,143240,189860,192250,214430,219130
date,,,,,,,,,,,,,,,,,,,,,
2020-03-31,6.700367e+10,2.855757e+10,7.399416e+10,5.973060e+11,5.338715e+11,3.132761e+11,2.515361e+12,5.315096e+11,4.427092e+12,4.017133e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-06-30,6.983330e+10,3.120514e+10,8.620966e+10,5.794880e+11,5.815788e+11,4.154865e+11,2.649998e+12,5.796918e+11,4.209198e+12,4.227164e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-09-30,6.656821e+10,2.908280e+10,8.432171e+10,6.450780e+11,6.242564e+11,4.296809e+11,2.774502e+12,6.216856e+11,4.199465e+12,4.068392e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-12-31,6.867021e+10,2.898887e+10,8.469921e+10,6.493540e+11,5.166162e+11,4.614219e+11,2.841267e+12,5.141190e+11,4.244994e+12,4.778968e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-03-31,7.182724e+10,2.270011e+10,8.257075e+10,7.006650e+11,5.350730e+11,3.790340e+11,2.692603e+12,5.324351e+11,4.520269e+12,4.511859e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-06-30,7.580235e+10,3.799214e+10,9.396038e+10,7.852480e+11,5.654802e+11,4.333381e+11,2.747235e+12,5.626648e+11,3.512257e+12,5.276865e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-09-30,6.907114e+10,3.450467e+10,9.347672e+10,8.214320e+11,5.574221e+11,4.514674e+11,2.846489e+12,5.549178e+11,3.789304e+12,4.804644e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-12-31,7.631738e+10,3.831351e+10,1.133493e+11,7.999680e+11,5.449286e+11,4.239708e+11,3.057325e+12,NaN,1.382324e+12,5.200663e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-03-31,8.529413e+10,2.915054e+10,9.678620e+10,8.222970e+11,5.836753e+11,4.108533e+11,2.856953e+12,NaN,3.419515e+12,5.301244e+10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
revenue_actual_forecast[['000660']]

,000660
date,
2020-03-31,7.198892e+12
2020-06-30,8.606534e+12
2020-09-30,8.128767e+12
2020-12-31,7.966225e+12
2021-03-31,8.494188e+12
2021-06-30,1.032167e+13
2021-09-30,1.180532e+13
2021-12-31,1.237661e+13
2022-03-31,1.215565e+13


In [12]:
# 0) 원본을 건드리지 않도록 복사
df = revenue_actual_forecast.copy()

print("원본 컬럼 수:", df.shape[1])
df_clean = df.dropna(axis=1, how='any')
print("NaN 제거 후 컬럼 수:", df_clean.shape[1])

# 1) 인덱스를 datetime 으로 확실히 변환
df_clean.index = pd.to_datetime(df_clean.index)

# 2) index에서 연도 추출하여 year 컬럼 생성
df_clean['year'] = df_clean.index.year

# 3) 연도별 groupby 후 합계 계산 → 연간 매출 데이터 생성
annual_revenue = df_clean.groupby('year').sum()

# 4) year 컬럼 제거 (원본 형태 유지 목적)
annual_revenue.index.name = 'year'

# 3) 연간 성장률(전년 대비, YoY) 계산
#    결과: 연도 x ticker, 값은 성장률(비율)
annual_growth = annual_revenue.pct_change()

# 4) 2026년 매출 성장률만 추출
#    (2025년, 2026년 데이터가 있어야 값이 나옴)
growth_2026 = annual_growth.loc[2026].dropna()   # index=티커, 값=2026년 성장률

# 5) 성장률 높은 순으로 정렬
growth_2026_sorted = growth_2026.sort_values(ascending=False)

# 6) 보기 좋게 DataFrame 으로 정리 + 순위 컬럼 추가
growth_2026_rank = (
    growth_2026_sorted
    .to_frame(name="growth_2026")  # 열 이름 지정
)
growth_2026_rank["rank"] = growth_2026_rank["growth_2026"].rank(
    ascending=False, method="dense"
).astype(int)

# 7) 결과 확인 (상위 30개 예시)
print(growth_2026_rank.head(30))

원본 컬럼 수: 2401
NaN 제거 후 컬럼 수: 880
        growth_2026  rank
052300     2.127576     1
046390     1.587682     2
005110     1.131908     3
066970     1.045599     4
000300     1.000003     5
001140     0.949958     6
065170     0.756695     7
064090     0.652531     8
000660     0.498991     9
064800     0.442538    10
059090     0.393686    11
065770     0.378833    12
030350     0.362502    13
012450     0.359760    14
002760     0.351125    15
048530     0.301029    16
095660     0.295880    17
025980     0.291623    18
003230     0.286203    19
079810     0.281605    20
050120     0.271379    21
032500     0.262198    22
064350     0.261844    23
019180     0.254437    24
012320     0.237308    25
035900     0.236125    26
001420     0.235052    27
031980     0.221371    28
040160     0.219654    29
009540     0.219403    30


In [26]:
growth_2026_rank.iloc[60:100]

,growth_2026,rank
000500,0.133649,61
001390,0.133431,62
007460,0.132127,63
054920,0.131953,64
038460,0.127348,65
036800,0.126858,66
054210,0.126500,67
033500,0.125889,68
035890,0.125379,69
019010,0.125136,70


In [18]:
from DATA.ks_marketcap_fetcher import fetch_marketcap_for_tickers

tickers = growth_2026_rank.index.tolist()
tics = tickers[:30]
df = fetch_marketcap_for_tickers(tickers)
print(df)

           market_cap
ticker               
A052300  3.811520e+10
A046390  5.146210e+10
A005110  2.646740e+10
A066970  4.362710e+12
A000300  1.644220e+11
...               ...
A048410  5.051750e+11
A079950  7.957600e+09
A000040  4.275570e+10
A008700  9.663740e+10
A025620  2.169910e+11

[880 rows x 1 columns]


In [24]:
df.iloc[70:100]

,market_cap
ticker,
A005670,6.240000e+10
A044180,1.603070e+10
A040350,7.046910e+10
A017370,1.137300e+11
A058450,1.099170e+10
A037270,4.182670e+11
A011320,9.186990e+10
A033100,5.774440e+11
A065690,1.714020e+10
